# Task 1：实现 LIF 神经元

目标：模拟**一个神经元**接收恒定电流后，膜电位如何变化、何时发放脉冲。依次运行下方单元格即可。

## 1. LIF 神经元模型

LIF 的本质是：**输入电流改变膜电位，泄漏使膜电位趋向静息值**。发放和重置前，膜电位满足：

$$\tau_m\frac{dV(t)}{dt}=-(V(t)-V_{\mathrm{rest}})+RI(t)$$

其中：

- $V$ 和 $V_{\mathrm{rest}}$：当前膜电位和静息电位；
- $\frac{dV}{dt}$：膜电位随时间变化的速度；
- $-(V-V_{\mathrm{rest}})$：泄漏，使膜电位趋向静息值；
- $RI(t)$：输入电流的作用；
- $\tau_m$：膜时间常数，影响电位变化的快慢。

令 $k$ 为时间步编号，$t_k=k\Delta t$。使用前向 Euler 方法，将连续变化近似为离散更新：

$$V_{k+1}=V_k+\frac{\Delta t}{\tau_m}\left[-(V_k-V_{\mathrm{rest}})+RI_k\right]$$

达到阈值后的脉冲发放、重置和不应期是额外规则。下面的 `LIFNeuron` 在内部适配 `snn.Leaky`，对外使用题目中的物理参数。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch

## 2. 模拟一个神经元

`LIFNeuron` 接受电流（nA），返回膜电位（mV）和脉冲 Tensor。达到阈值时发放并立即重置，随后 2 ms 不应期内保持重置电位、不发放。使用 snnTorch 代理梯度让脉冲在反向传播时可近似求导；不应期仍是离散状态。

In [ ]:
# 电位 mV、时间 ms、电阻 MΩ、电流 nA；dt 和 tau_m 对应公式中的 Δt 和 τₘ
V_rest, V_reset, V_th = -65.0, -65.0, -50.0
dt, tau_m, R, I = 0.1, 20.0, 10.0, 2.0
refractory_ms, duration_ms = 2.0, 2000.0

In [ ]:
# 前向达到阈值即发放；反向使用 snnTorch 的代理梯度。
surrogate = snn.surrogate.fast_sigmoid()


def at_threshold(voltage_from_threshold):
    return surrogate(voltage_from_threshold) + (voltage_from_threshold == 0).to(
        voltage_from_threshold.dtype
    )


class LIFNeuron(torch.nn.Module):
    def __init__(self, dt, tau_m, R, V_rest, V_reset, V_th, refractory_ms):
        super().__init__()
        self.dt, self.tau_m, self.R = dt, tau_m, R
        self.V_rest, self.V_reset = V_rest, V_reset
        self.refractory_steps = round(refractory_ms / dt)
        self.neuron = snn.Leaky(
            beta=torch.tensor(self._decay_factor(), dtype=torch.float32),
            threshold=self._relative_voltage(V_th),
            reset_mechanism="zero",
            reset_delay=False,
            spike_grad=at_threshold,
        )
        self.register_buffer(
            "U",
            torch.tensor(self._relative_voltage(V_reset), dtype=torch.float32),
            persistent=False,
        )
        self.register_buffer(
            "remaining", torch.zeros((), dtype=torch.long), persistent=False
        )

    def _relative_voltage(self, V):
        return V - self.V_rest

    def _decay_factor(self):
        return 1 - self.dt / self.tau_m

    def _input_increment(self, I):
        return self.dt / self.tau_m * self.R * I

    def reset_state(self):
        self.U = self.U.new_tensor(self._relative_voltage(self.V_reset))
        self.remaining = torch.zeros_like(self.remaining)

    def forward(self, I):
        I = torch.as_tensor(I, dtype=self.U.dtype, device=self.U.device)
        X = self._input_increment(I)
        spike, U = self.neuron(X, self.U)
        active = self.remaining == 0
        spike = torch.where(active, spike, torch.zeros_like(spike))
        U_reset = self.U.new_tensor(self._relative_voltage(self.V_reset))
        self.U = torch.where(active, U + spike * U_reset, U_reset)
        self.remaining = torch.where(
            active,
            spike.detach().to(torch.long) * self.refractory_steps,
            self.remaining - 1,
        )
        return self.U + self.V_rest, spike

In [ ]:
def simulate(I, duration_ms, dt):
    neuron = LIFNeuron(dt, tau_m, R, V_rest, V_reset, V_th, refractory_ms)
    voltages, spikes = [], []
    for _ in range(round(duration_ms / dt)):
        V, spike = neuron(I)
        voltages.append(V)
        spikes.append(spike)
    # 只在绘图前将 Tensor 转换为 NumPy。
    return torch.stack(voltages).detach().cpu().numpy(), torch.stack(
        spikes
    ).detach().cpu().numpy()

## 3. 观察一次仿真

给神经元施加 2 nA 恒定电流，运行 2 秒。图中只展示前 200 ms，红虚线是发放阈值，圆点是脉冲发生的时刻。这里记录的是**每一步结束后的电位**，因此脉冲对应的电位已经重置。

In [ ]:
voltages, spikes = simulate(I=I, duration_ms=duration_ms, dt=dt)
time_ms = (np.arange(len(voltages)) + 1) * dt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_ms, voltages, label="Membrane potential")
ax.axhline(V_th, color="red", linestyle="--", label="Threshold")
ax.scatter(
    time_ms[spikes.astype(bool)],
    voltages[spikes.astype(bool)],
    s=8,
    label="Spike / reset",
)
ax.set(xlabel="Time (ms)", ylabel="Voltage (mV)")
ax.set_xlim(0, 200)
ax.legend()
plt.show()
print(
    f"脉冲数：{int(spikes.sum())}；平均发放率：{spikes.sum() / (len(spikes) * dt / 1000):.1f} Hz"
)

## 4. 用理论检查实现

默认参数下，临界恒定电流为 $(V_{th}-V_{rest})/R=1.5$ nA。低于临界值时电位达不到阈值；恰好等于临界值时电位渐近接近阈值，在有限时间内也不发放。将上面的 2 nA 改为 1 nA、1.5 nA 再运行，观察脉冲是否消失。

这里完成的是 Task 1。不同输入、系统对照实验和滑动发放率属于后续任务。